# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display general description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use `metadata.record_sets` to list the dataset contents. Each record set, field, and column is referenced by its `@id`.

In [ ]:
record_sets = dataset.metadata.record_sets

print("Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')}")

# Show fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    fields = rs.get('fields', [])
    if fields:
        print("Fields:")
        for f in fields:
            print(f"  - @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
            # Optionally print columns
            columns = f.get('columns', [])
            if columns:
                print("    Columns:")
                for c in columns:
                    print(f"      - @id: {c['@id']}, name: {c.get('name', 'N/A')}, dataType: {c.get('dataType', 'N/A')}")
    else:
        print("No fields defined for this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All record sets and fields are referenced by their `@id` values, as seen above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}

# Load data from each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display columns of the main record set (choose first for demonstration)
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We reference all fields/columns by their `@id`.

For demonstration, if there is an `age` or numeric field present, we use its `@id` for operations. You can adjust the field selection to your dataset.

In [ ]:
# Pick the main record set for EDA
df = None
if len(dataframes) > 0:
    df = dataframes[main_record_set_id]
else:
    print("No dataframe loaded.")

# Determine a numeric field by inspecting field definitions
numeric_field_id = None
group_field_id = None
for rs in dataset.metadata.record_sets:
    if rs['@id'] == main_record_set_id:
        for f in rs.get('fields', []):
            # Example: search for age or similar numeric fields
            if f.get('dataType') in ['Integer', 'Float', 'Number']:
                numeric_field_id = f['@id']
            # Example: group by sex or msi_status
            if f.get('name', '').lower() in ['sex', 'msi_status', 'anatomical_location']:
                group_field_id = f['@id']
        break

# If there is no numeric field detected, try from dataframe columns
if df is not None and numeric_field_id is None:
    # Choose the first numeric column
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except:
            continue

if numeric_field_id is not None:
    # Threshold (arbitrarily chosen as 10, you may adjust per dataset)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Use referenced columns/fields for plotting. Example: Plot histogram for numeric field, or bar plot grouped by category.

In [ ]:
# Visualization
if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id is not None and group_field_id in df.columns and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides structured clinical and biomarker data for second primary colorectal cancer in survivors.
- Exploration showed key field and record set structure using `@id` referencing.
- Numeric analyses and visualizations can be conducted using the detected fields and columns.
- The data is well-structured for ML/data science tasks, and all entities can be referenced reliably by their `@id`.